# Set up the Lakehouse target table and watermark (explained)

Attach `lh_meridian_hr` as the default lakehouse, then run this notebook as the **first** activity of the event-ingestion pipeline. It does two idempotent things:

1. Pre-creates the shared `bronze.workforce_events_raw` Delta table so the parallel Copy activities append to a table that already exists.
2. Creates `bronze.ingestion_watermark`, seeds it with `default_watermark` the first time only, and **exits the current watermark** so the next step receives it as a parameter.

Because this notebook owns watermark initialization, the downstream discovery notebook no longer creates or reads the control table — it simply uses the watermark it is handed.

> This is an annotated copy of `nb_setup_lakehouse.ipynb`. Each code cell is preceded by a **Summary** and a collapsible **line-by-line** explanation.

## Parameters (mark this as the parameter cell)

**Summary.** Identifies which pipeline's watermark this run owns and the value to seed on a first run.

<details>
<summary>Line-by-line details</summary>

- `pipeline_name = "workforce_events"` — the key that identifies this pipeline's row in `bronze.ingestion_watermark`.
- `default_watermark = "2020-12-01 00:00:00"` — seeded only when no row exists yet; a deliberately old month so the first load picks up every file.

</details>

In [ ]:
pipeline_name = "workforce_events"
default_watermark = "2020-12-01 00:00:00"

## Create the schema and pre-create the target table

**Summary.** Ensure the `bronze` schema exists and pre-create the empty `bronze.workforce_events_raw` Delta table with an explicit column layout, so the pipeline's parallel Copy activities append to a table that already exists instead of racing to create it.

<details>
<summary>Line-by-line details</summary>

- `spark.sql("CREATE SCHEMA IF NOT EXISTS bronze")` — creates the `bronze` schema (database) if it is not already present; harmless on re-runs.
- `CREATE TABLE IF NOT EXISTS bronze.workforce_events_raw (...)` — defines the raw events table only when it is missing, which makes the DDL idempotent (safe to run repeatedly).
- The column list fixes the schema up front: identifiers (`event_id`, `employee_id`, `cost_center_id`), the `event_date`, classification (`classification_group`, `classification_level`), the `event_type`, money (`amount_local` as `DECIMAL(18, 2)` plus `local_currency`), `work_country_code`, and lineage columns (`ingest_ts`, `source_system`).
- `USING DELTA` — stores the table as a Delta Lake table (Parquet files plus a transaction log), so appends are ACID.
- `print("bronze.workforce_events_raw is ready")` — writes a confirmation line to the cell output.

</details>

In [ ]:
spark.sql("CREATE SCHEMA IF NOT EXISTS bronze")

spark.sql("""
CREATE TABLE IF NOT EXISTS bronze.workforce_events_raw (
    event_id STRING,
    event_date DATE,
    employee_id STRING,
    cost_center_id STRING,
    classification_group STRING,
    classification_level INT,
    event_type STRING,
    amount_local DECIMAL(18, 2),
    local_currency STRING,
    work_country_code STRING,
    ingest_ts TIMESTAMP,
    source_system STRING
)
USING DELTA
""")

print("bronze.workforce_events_raw is ready")

## Create, seed, and return the watermark

**Summary.** Creates the `bronze.ingestion_watermark` control table if it is missing, seeds `default_watermark` only on a first run, then reads the stored value back and exits it so the next pipeline step receives the watermark instead of deriving it.

<details>
<summary>Line-by-line details</summary>

- The two guards reject an empty `pipeline_name` and a `default_watermark` that is not the first day of a month.
- `CREATE TABLE IF NOT EXISTS bronze.ingestion_watermark (...)` — idempotently creates the control table (`pipeline_name`, `watermark_timestamp`, `updated_at`).
- `spark.createDataFrame([...])` + `createOrReplaceTempView("watermark_seed")` — builds the one-row source for the MERGE.
- The MERGE uses **`WHEN NOT MATCHED` only**, so it inserts the default on a first run and leaves an existing watermark untouched on every later run.
- The `spark.table(...).where(col("pipeline_name") == pipeline_name)` read returns the row that is now guaranteed to exist — the freshly seeded default on a first run, or the stored watermark afterwards.
- `notebookutils.notebook.exit(json.dumps(result))` — returns `pipeline_name` and `watermark` to the pipeline, which passes the watermark into the file-discovery notebook.

</details>

In [ ]:
import json
from datetime import datetime

from pyspark.sql.functions import col

if not pipeline_name.strip():
    raise ValueError("pipeline_name must not be empty")

parsed_default = datetime.strptime(default_watermark, "%Y-%m-%d %H:%M:%S")
if parsed_default.day != 1:
    raise ValueError("default_watermark must be the first day of a month")

spark.sql("""
CREATE TABLE IF NOT EXISTS bronze.ingestion_watermark (
    pipeline_name STRING,
    watermark_timestamp TIMESTAMP,
    updated_at TIMESTAMP
)
USING DELTA
""")

watermark_seed = spark.createDataFrame(
    [(pipeline_name, parsed_default)],
    "pipeline_name STRING, watermark_timestamp TIMESTAMP",
)
watermark_seed.createOrReplaceTempView("watermark_seed")

# Insert-only: an existing watermark is never overwritten.
spark.sql("""
MERGE INTO bronze.ingestion_watermark AS target
USING watermark_seed AS source
ON target.pipeline_name = source.pipeline_name
WHEN NOT MATCHED THEN INSERT (pipeline_name, watermark_timestamp, updated_at)
    VALUES (source.pipeline_name, source.watermark_timestamp, current_timestamp())
""")

watermark_row = (
    spark.table("bronze.ingestion_watermark")
    .where(col("pipeline_name") == pipeline_name)
    .select("watermark_timestamp")
    .collect()
)
watermark = watermark_row[0]["watermark_timestamp"].strftime("%Y-%m-%d %H:%M:%S")

result = {
    "pipeline_name": pipeline_name,
    "watermark": watermark,
}
print(result)
notebookutils.notebook.exit(json.dumps(result))